# 04 — Create Accounts

Mirrors the original per-customer account-creation notebook
(`OneBill_Customer_Migration.ipynb`): pulls every `vBill` account from
`bi_datastore.billing_account`, cleans stray suffixes out of the name
(`(BOND)`, `(LIQUIDATION)`, etc.), classifies each as Individual (`1001`)
or Business (`1002`), attaches every Dataverse contact on the account, and
creates it in OneBill — idempotently, in parallel.

**This is written for a full bulk migration** (every vBill account, not
just a test subset) — set `TEST_ROW_LIMIT = None` when you're ready.

## The two Williams "bucket" accounts

This particular migration also needs two accounts that **aren't** real
vBill accounts — `Managed by Williams` and `Williams Corporation` — which
every subscription in `05_Fetch_Subscriptions.ipynb` onward gets routed
to. Rather than special-case those two accounts outside this pipeline,
they're just two more rows in `MANUAL_BUCKET_ACCOUNTS` below, built by
hand instead of pulled from MySQL, and go through the exact same
build-payload / create / idempotency logic as everything else.

**Need a second "Managed by Williams" account later?** Add another dict to
`MANUAL_BUCKET_ACCOUNTS` — no other changes required to create it. (Note:
`onebill_common.TARGET_ACCOUNTS` — used by `05_Fetch_Subscriptions.ipynb`
to *route* subscriptions — only knows about the original two account keys,
so a third manual account here gets created but won't automatically
receive any subscriptions; that routing would need its own decision.)

**Idempotency** — same three-way contract as the original: `"created"` /
`"exists"` (not a failure — proceed as normal) / `"failed"`.

## 1. Setup

In [1]:
#%pip install openpyxl

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from sqlalchemy import create_engine
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_accounts")
session = new_session(max_workers=MAX_WORKERS)

# Use this to limit rows while testing. Set to None once ready for a full run.
TEST_ROW_LIMIT = 5
BATCH_NUMBER =  os.environ["BATCH_NUMBER"]

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 12
python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 23
python-dotenv could not parse statement starting at line 29


## 2. Load contacts (from `01_Fetch_Contacts.ipynb`)

In [2]:
df_contacts_raw = load_df("contacts_raw")
contacts_by_account = index_contacts_by_account(df_contacts_raw)
logger.info(
    f"Indexed {sum(len(v) for v in contacts_by_account.values()):,} contacts "
    f"across {len(contacts_by_account):,} accounts"
)


2026-07-22 05:36:15,174 [INFO] Indexed 41,880 contacts across 37,552 accounts


## 3. Bulk account query

Unchanged from the original notebook's account-cleaning logic. No
`AccountCode` filter — this pulls every `vBill` account.

`AccountCode_Batch` is the `accountNumber` actually sent to OneBill —
`AccountCode` plus `ACCOUNT_NUMBER_SUFFIX` (blank by default; set it in
`.env` for a test run so you don't collide with real account numbers).

In [3]:
assert BI_DATASTORE_URL, "DB_USERNAME/DB_PASSWORD/DB_HOST not set in .env"
engine = create_engine(BI_DATASTORE_URL)

ACCOUNT_QUERY = """
SELECT
    `AccountName` AS `AccountName_Original`
    ,TRIM(
		REPLACE(
			REPLACE(
				REPLACE(
					REPLACE(
						REPLACE(
							REPLACE(
								REPLACE(
									REPLACE(
										REPLACE(
											REPLACE(
												REPLACE(
													REPLACE(
														REPLACE(
															REPLACE(
																REPLACE(
																	REPLACE(
																		REPLACE(
																			REPLACE(
																				REPLACE(
																					REPLACE(`AccountName`, '(BOND - DECLINED)', ''),
																				'(BOND)', ''),
																			'(ICMS)', ''),
																		'(Staff)', ''),
																	'(X)', ''),
																'(In Liquidation)', ''),
															'zz-', ''),
														'(DECLINED)', ''),
													'(Operator)', ''),
												'(Liquidation )', ''),
											'(LIQUIDATION)', ''),
										'(Under Liquidation)', ''),
									'[LIQUIDATION]', ''),
								'(Bad - Debt)', ''),
							'(Bad Debt)', ''),
						'(Bad-Debt)', ''),
					'(BOND - DECLINED)', ''),
				'(BOND DECLINED)', ''),
			'(BOND-DECLINED)', ''),
        '(COMPRIMISED)', '') -- Even though it is spelt incorrectly there are no '(COMPROMISED)' accounts, only '(COMPRIMISED)', so we need to catch this too
    ) AS `AccountName_Cleaned`
    ,`AccountCode`
    ,`CreatedDate`
    ,`ClosedDate`
    ,`AccountType`
    ,CASE
		WHEN `AccountType` IN ('Residential', 'Actrix Residential', 'Consumer', 'Standard Account', 'Internal-Use Account', 'Staff')
        AND `AccountName` NOT LIKE '%Ltd%'
        AND `AccountName` NOT LIKE '%Limited%'
        AND `AccountName` NOT LIKE '%Pty%'
		THEN '1001' -- Individual Customer
        ELSE '1002' -- Business Customer 
	END AS `OneBill_AccountType`
    ,CASE
        WHEN `_temporary_crmonly_addr1` IS NULL OR `_temporary_crmonly_addr1` = '' THEN '1 Somewhere Place'
        ELSE `_temporary_crmonly_addr1`
    END AS `Address1`
    ,`_temporary_crmonly_addr2` AS `Address2`
    ,`_temporary_crmonly_suburb` AS `Suburb`
    ,CASE
        WHEN `_temporary_crmonly_city` IS NULL OR `_temporary_crmonly_city` = '' THEN 'Auckland'
        ELSE `_temporary_crmonly_city`
    END AS `City`
    ,CASE
        WHEN `_temporary_crmonly_postcode` IS NULL OR `_temporary_crmonly_postcode` = '' THEN '0001'
        ELSE `_temporary_crmonly_postcode`
    END AS `Postcode`
    ,`_temporary_crmonly_dob` AS `DateOfBirth`
FROM
    bi_datastore.billing_account
WHERE
    `_DataSource` = 'vBill'
AND 
	`AccountCode` = '99965692'
ORDER BY
    `AccountCode` DESC
""".strip()

df_mysql_accounts = pd.read_sql(ACCOUNT_QUERY, con=engine)
df_mysql_accounts["AccountCode"] = df_mysql_accounts["AccountCode"].astype(str)

df_mysql_accounts["AccountCode_Batch"] = df_mysql_accounts["AccountCode"] + BATCH_NUMBER

df_mysql_accounts["AccountName_Unique"] = (
    df_mysql_accounts["AccountName_Cleaned"] + " (" + df_mysql_accounts["AccountCode_Batch"] + ")"
)

logger.info(f"Loaded {len(df_mysql_accounts):,} accounts from MySQL")

if TEST_ROW_LIMIT is not None:
    df_mysql_accounts = df_mysql_accounts.head(TEST_ROW_LIMIT)  # Testing limiter — remove/raise for a full run.
    logger.info(f"TEST_ROW_LIMIT active — trimmed to {len(df_mysql_accounts):,} rows")

df_mysql_accounts.head()


2026-07-22 05:36:21,036 [INFO] Loaded 1 accounts from MySQL
2026-07-22 05:36:21,038 [INFO] TEST_ROW_LIMIT active — trimmed to 1 rows


,AccountName_Original,AccountName_Cleaned,AccountCode,CreatedDate,ClosedDate,AccountType,OneBill_AccountType,Address1,Address2,Suburb,City,Postcode,DateOfBirth,AccountCode_Batch,AccountName_Unique
0,Williams Internet Limited,Williams Internet Limited,99965692,2025-11-19,None,Wholesale Unlimited,1002,124 Peterborough Street\nChristchurch Central ...,None,None,Auckland,0001,None,99965692_10001,Williams Internet Limited (99965692_10001)


## 4. Manually-defined bucket accounts

Not sourced from MySQL — built by hand, same column shape as the bulk
query above so `build_account_payload` treats them identically. Add more
dicts here any time (e.g. a second "Managed by Williams" account).

In [4]:
TODAY = datetime.now().strftime("%Y-%m-%d")

MANUAL_BUCKET_ACCOUNTS = [
    {
        "AccountKey":            "managed_by_williams",  # <-- lets 05_Fetch_Subscriptions.ipynb find this row by role, not just by name/number
        "AccountName_Original":  "Managed by Williams",
        "AccountName_Cleaned":   "Managed by Williams",
        "AccountName_Unique":    "Managed by Williams" + " (" + df_mysql_accounts["AccountCode_Batch"].iloc[0] + "1" + ")",
        "AccountCode":           df_mysql_accounts["AccountCode"].iloc[0] + "1",
        "AccountCode_Batch":     df_mysql_accounts["AccountCode"].iloc[0] + "1" + BATCH_NUMBER,  # no ACCOUNT_NUMBER_SUFFIX — this IS the real target account number
        "CreatedDate":           TODAY,
        "ClosedDate":            None,
        "AccountType":           "Wholesale Unlimited",
        "OneBill_AccountType":   "1002",  # Business
        "Address1":              "1 Somewhere Place",
        "Address2":              None,
        "Suburb":                None,
        "City":                  "Auckland",
        "Postcode":              "0001",
        "DateOfBirth":           None,
    },
    # {
    #     "AccountKey":             "williams_corporation",
    #     "AccountName_Original":  TARGET_ACCOUNTS["williams_corporation"]["account_name"],
    #     "AccountName_Cleaned":   TARGET_ACCOUNTS["williams_corporation"]["account_name"],
    #     "AccountName_Unique":    TARGET_ACCOUNTS["williams_corporation"]["account_name"],
    #     "AccountCode":           TARGET_ACCOUNTS["williams_corporation"]["account_number"],
    #     "AccountCode_Batch":     TARGET_ACCOUNTS["williams_corporation"]["account_number"],
    #     "CreatedDate":           TODAY,
    #     "ClosedDate":            None,
    #     "AccountType":           "Migration Bucket Account",
    #     "OneBill_AccountType":   "1002",  # Business
    #     "Address1":              "1 Somewhere Place",
    #     "Address2":              None,
    #     "Suburb":                None,
    #     "City":                  "Auckland",
    #     "Postcode":              "0001",
    #     "DateOfBirth":           None,
    # },
    # Add more here, e.g. a second "Managed by Williams" account:
    # {
    #     "AccountKey":            "managed_by_williams",  # reuse the key so 05 knows this is also a valid target for that routing rule — see note below
    #     "AccountName_Original": "Managed by Williams 2",
    #     "AccountName_Cleaned":  "Managed by Williams 2",
    #     "AccountName_Unique":   "Managed by Williams 2",
    #     "AccountCode":          "MANAGED-BY-WILLIAMS-2",
    #     "AccountCode_Batch":    "MANAGED-BY-WILLIAMS-2",
    #     "CreatedDate":          TODAY,
    #     "ClosedDate":           None,
    #     "AccountType":          "Migration Bucket Account",
    #     "OneBill_AccountType":  "1002",
    #     "Address1":             "1 Somewhere Place",
    #     "Address2":             None,
    #     "Suburb":               None,
    #     "City":                 "Auckland",
    #     "Postcode":             "0001",
    #     "DateOfBirth":          None,
    # },
]

df_manual_accounts = pd.DataFrame(MANUAL_BUCKET_ACCOUNTS)
df_manual_accounts


,AccountKey,AccountName_Original,AccountName_Cleaned,AccountName_Unique,AccountCode,AccountCode_Batch,CreatedDate,ClosedDate,AccountType,OneBill_AccountType,Address1,Address2,Suburb,City,Postcode,DateOfBirth
0,managed_by_williams,Managed by Williams,Managed by Williams,Managed by Williams (99965692_100011),999656921,999656921_10001,2026-07-22,None,Wholesale Unlimited,1002,1 Somewhere Place,None,None,Auckland,0001,None


## 5. Combine — one account list, one pipeline

In [5]:
df_accounts = pd.concat([df_mysql_accounts, df_manual_accounts], ignore_index=True)
logger.info(f"{len(df_accounts):,} accounts total ({len(df_mysql_accounts):,} from MySQL + {len(df_manual_accounts):,} manual bucket accounts)")


2026-07-22 05:36:21,114 [INFO] 2 accounts total (1 from MySQL + 1 manual bucket accounts)


## 6. Per-account worker

In [6]:
def migrate_account(row: dict, session: requests.Session, contacts_by_account: dict[str, list[dict]]) -> dict:
    account_code = row["AccountCode"]
    account_name = row["AccountName_Unique"]

    payload = build_account_payload(row, contacts_by_account)

    status, error, onebill_id = "failed", None, None
    try:
        status, response, message = create_onebill_account(session, payload)
        if status == "created":
            onebill_id = (response or {}).get("accountId", "unknown")
            logger.info(f"[OK] {account_code} -> {row['AccountCode_Batch']} (OneBill id={onebill_id})")
        elif status == "exists":
            error = message
            logger.info(f"[EXISTS] {account_code} -> {row['AccountCode_Batch']} already in OneBill")
        else:
            error = message
            logger.error(f"[FAIL] {account_code} -> {row['AccountCode_Batch']} — {error}")
    except Exception as e:
        error = str(e)
        logger.error(f"[FAIL] {account_code} -> {row['AccountCode_Batch']} — {error}")

    return {
        "AccountCode":       account_code,
        "AccountCode_Batch": row["AccountCode_Batch"],
        "AccountName":       account_name,
        "AccountKey":        clean(row.get("AccountKey")),  # None for bulk MySQL rows; e.g. "managed_by_williams" for manual bucket rows
        "status":            status,
        "onebill_id":        onebill_id,
        "error":             error,
    }


## 7. Run (parallel driver)

In [7]:
def create_all_accounts(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    rows = df.to_dict("records")
    total = len(rows)
    results = []

    logger.info(f"Creating {total:,} accounts with {max_workers} workers...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(migrate_account, row, session, contacts_by_account): row["AccountCode"]
            for row in rows
        }
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] in ("created", "exists"))
                logger.info(f"Progress: {i}/{total} — {ok} created/existing so far")

    return pd.DataFrame(results)


df_account_results = create_all_accounts(df_accounts)
df_account_results.head(20)


2026-07-22 05:36:21,240 [INFO] Creating 2 accounts with 10 workers...
2026-07-22 05:36:29,354 [INFO] [OK] 99965692 -> 99965692_10001 (OneBill id=unknown)
2026-07-22 05:36:29,509 [INFO] [OK] 999656921 -> 999656921_10001 (OneBill id=unknown)
2026-07-22 05:36:29,514 [INFO] Progress: 2/2 — 2 created/existing so far


,AccountCode,AccountCode_Batch,AccountName,AccountKey,status,onebill_id,error
0,99965692,99965692_10001,Williams Internet Limited (99965692_10001),None,created,unknown,None
1,999656921,999656921_10001,Managed by Williams (99965692_100011),managed_by_williams,created,unknown,None


## 8. Save + failure summary

In [8]:
save_df("account_results", df_account_results)

failures = df_account_results[df_account_results["status"] == "failed"]
print(f"{len(failures):,} / {len(df_account_results):,} accounts failed to create")
failures.head(20)


Saved 2 rows -> migration_data\04_account_creation_results.csv
0 / 2 accounts failed to create


,AccountCode,AccountCode_Batch,AccountName,AccountKey,status,onebill_id,error


## 9. Stop here if either **bucket** account failed

Bulk-account failures are just reported above — a full migration
shouldn't halt on one bad row. The two Williams bucket accounts are
different: `05_Fetch_Subscriptions.ipynb` onward hard-depends on both
existing, so fail loudly here if either didn't make it.

In [9]:
bucket_account_numbers = {v["account_number"] for v in TARGET_ACCOUNTS.values()}
bucket_failures = df_account_results[
    df_account_results["AccountCode_Batch"].isin(bucket_account_numbers)
    & (df_account_results["status"] == "failed")
]

if not bucket_failures.empty:
    raise RuntimeError(
        f"{len(bucket_failures)} bucket account(s) failed to create — see 'error' column above before proceeding.\n"
        f"{bucket_failures[['AccountCode_Batch', 'error']].to_string()}"
    )
print("Both Williams bucket accounts are ready in OneBill.")


Both Williams bucket accounts are ready in OneBill.
